In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = PROCESSED_DIR / "market_data_cleaned.csv"

csv_files = list(RAW_DIR.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")
print(f"Output file: {OUTPUT_FILE}")

CSV files found: 325
Output file: ..\data\processed\market_data_cleaned.csv


In [3]:
COLUMN_RENAME = {
    "Arrivals": "Arrivals (Tonnes)",
    "Min Price": "Min Price (Rs./Quintal)",
    "Max Price": "Max Price (Rs./Quintal)",
    "Modal Price": "Modal Price (Rs./Quintal)"
}

STANDARD_COLUMNS = [
    "State Name",
    "District Name",
    "Market Name",
    "Variety",
    "Group",
    "Arrivals (Tonnes)",
    "Min Price (Rs./Quintal)",
    "Max Price (Rs./Quintal)",
    "Modal Price (Rs./Quintal)",
    "Reported Date",
    "Commodity"
]

print("Column normalization rules ready.")

Column normalization rules ready.


In [4]:
test_file = next(
    file for file in csv_files
    if file.stat().st_size > 0
)

test_df = pd.read_csv(
    test_file,
    encoding="utf-8-sig",
    on_bad_lines="skip"
)

test_df = test_df.rename(
    columns=COLUMN_RENAME
)

test_df["Commodity"] = test_file.stem

print("File:", test_file.name)
print("Shape:", test_df.shape)

display(test_df.head())

File: Ajwan.csv
Shape: (27964, 11)


,State Name,District Name,Market Name,Variety,Group,Arrivals (Tonnes),Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal),Reported Date,Commodity
0,Andhra Pradesh,Kurnool,Adoni,Other,Spices,3.0,2811.0,2811.0,2811.0,2005-03-24,Ajwan
1,Andhra Pradesh,Kurnool,Adoni,Other,Spices,16.0,2800.0,3350.0,3211.0,2005-03-29,Ajwan
2,Andhra Pradesh,Kurnool,Adoni,Other,Spices,6.0,1300.0,3209.0,2700.0,2005-03-30,Ajwan
3,Andhra Pradesh,Kurnool,Adoni,Other,Spices,2.0,3419.0,3419.0,3419.0,2005-04-04,Ajwan
4,Andhra Pradesh,Kurnool,Adoni,Other,Spices,2.0,3001.0,3001.0,3001.0,2005-04-07,Ajwan


In [5]:
numeric_columns = [
    "Arrivals (Tonnes)",
    "Min Price (Rs./Quintal)",
    "Max Price (Rs./Quintal)",
    "Modal Price (Rs./Quintal)"
]

for column in numeric_columns:
    test_df[column] = pd.to_numeric(
        test_df[column],
        errors="coerce"
    )

test_df["Reported Date"] = pd.to_datetime(
    test_df["Reported Date"],
    errors="coerce"
)

print(test_df.dtypes)

State Name                              str
District Name                           str
Market Name                             str
Variety                                 str
Group                                   str
Arrivals (Tonnes)                   float64
Min Price (Rs./Quintal)             float64
Max Price (Rs./Quintal)             float64
Modal Price (Rs./Quintal)           float64
Reported Date                datetime64[us]
Commodity                               str
dtype: object


In [6]:
before = len(test_df)

test_df = test_df.dropna(
    subset=[
        "Reported Date",
        "Modal Price (Rs./Quintal)"
    ]
)

test_df = test_df[
    test_df["Modal Price (Rs./Quintal)"] >= 0
]

test_df = test_df[
    test_df["Max Price (Rs./Quintal)"] >=
    test_df["Min Price (Rs./Quintal)"]
]

after = len(test_df)

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning : {after:,}")
print(f"Rows removed        : {before - after:,}")

Rows before cleaning: 27,964
Rows after cleaning : 27,934
Rows removed        : 30


In [7]:
# Process all usable commodity files in chunks
# and create one cleaned dataset.

CHUNK_SIZE = 100_000

OUTPUT_FILE = PROCESSED_DIR / "market_data_cleaned.csv"

# Remove an old output file if it already exists
if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

processed_files = 0
processed_rows = 0
removed_rows = 0

for file in csv_files:

    # Skip empty files
    if file.stat().st_size == 0:
        continue

    print(f"Processing: {file.stem}")

    try:
        for chunk in pd.read_csv(
            file,
            chunksize=CHUNK_SIZE,
            encoding="utf-8-sig",
            on_bad_lines="skip"
        ):

            # Normalize the alternate schema
            chunk = chunk.rename(columns=COLUMN_RENAME)

            # Add commodity name
            chunk["Commodity"] = file.stem

            # Convert numeric columns
            for column in [
                "Arrivals (Tonnes)",
                "Min Price (Rs./Quintal)",
                "Max Price (Rs./Quintal)",
                "Modal Price (Rs./Quintal)"
            ]:
                chunk[column] = pd.to_numeric(
                    chunk[column],
                    errors="coerce"
                )

            # Convert date
            chunk["Reported Date"] = pd.to_datetime(
                chunk["Reported Date"],
                errors="coerce"
            )

            before = len(chunk)

            # Remove rows without essential information
            chunk = chunk.dropna(
                subset=[
                    "Reported Date",
                    "Modal Price (Rs./Quintal)"
                ]
            )

            # Remove invalid prices
            chunk = chunk[
                chunk["Modal Price (Rs./Quintal)"] >= 0
            ]

            # Ensure max price >= min price
            chunk = chunk[
                chunk["Max Price (Rs./Quintal)"] >=
                chunk["Min Price (Rs./Quintal)"]
            ]

            removed_rows += before - len(chunk)
            processed_rows += len(chunk)

            # Write the first chunk with headers,
            # then append subsequent chunks.
            chunk.to_csv(
                OUTPUT_FILE,
                mode="a",
                header=not OUTPUT_FILE.exists(),
                index=False
            )

        processed_files += 1

    except Exception as e:
        print(f"ERROR in {file.name}: {e}")

print("\n========== PROCESSING COMPLETE ==========")
print(f"Files processed : {processed_files}")
print(f"Rows retained   : {processed_rows:,}")
print(f"Rows removed    : {removed_rows:,}")
print(f"Output file     : {OUTPUT_FILE}")

Processing: Ajwan
Processing: Alasande Gram
Processing: Almond(Badam)
Processing: Alsandikai
Processing: Amaranthus
Processing: Ambada Seed
Processing: Amla(Nelli Kai)
Processing: Amphophalus
Processing: Antawala
Processing: Anthorium
Processing: Apple
Processing: Apricot(Jardalu-Khumani)
Processing: Arecanut(Betelnut-Supari)
Processing: Arhar (Tur-Red Gram)(Whole)
Processing: Arhar Dal(Tur Dal)
Processing: Ashgourd
Processing: Astera
Processing: Avare Dal
Processing: Bajra(Pearl Millet-Cumbu)
Processing: Balekai
Processing: Bamboo
Processing: Banana - Green
Processing: Banana
Processing: Barley (Jau)
Processing: Bay leaf (Tejpatta)
Processing: Beans
Processing: Beaten Rice
Processing: Beetroot
Processing: Bengal Gram Dal (Chana Dal)
Processing: Bengal Gram(Gram)(Whole)
Processing: Ber(Zizyphus-Borehannu)
Processing: Betal Leaves
Processing: Betelnuts
ERROR in Betelnuts.csv: No columns to parse from file
Processing: Bhindi(Ladies Finger)
Processing: Big Gram
Processing: Binoula
Process

In [8]:
# Cell 8 — Verify cleaned dataset

print("Output file exists:", OUTPUT_FILE.exists())

if OUTPUT_FILE.exists():
    file_size_gb = OUTPUT_FILE.stat().st_size / (1024 ** 3)
    print(f"Output file size: {file_size_gb:.2f} GB")

    # Read only a small sample
    sample_df = pd.read_csv(
        OUTPUT_FILE,
        nrows=10
    )

    print("\nShape of sample:", sample_df.shape)
    print("\nColumns:")
    print(sample_df.columns.tolist())

    print("\nSample rows:")
    display(sample_df)

Output file exists: True
Output file size: 4.63 GB

Shape of sample: (10, 11)

Columns:
['State Name', 'District Name', 'Market Name', 'Variety', 'Group', 'Arrivals (Tonnes)', 'Min Price (Rs./Quintal)', 'Max Price (Rs./Quintal)', 'Modal Price (Rs./Quintal)', 'Reported Date', 'Commodity']

Sample rows:


,State Name,District Name,Market Name,Variety,Group,Arrivals (Tonnes),Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal),Reported Date,Commodity
0,Andhra Pradesh,Kurnool,Adoni,Other,Spices,3.0,2811.0,2811.0,2811.0,2005-03-24,Ajwan
1,Andhra Pradesh,Kurnool,Adoni,Other,Spices,16.0,2800.0,3350.0,3211.0,2005-03-29,Ajwan
2,Andhra Pradesh,Kurnool,Adoni,Other,Spices,6.0,1300.0,3209.0,2700.0,2005-03-30,Ajwan
3,Andhra Pradesh,Kurnool,Adoni,Other,Spices,2.0,3419.0,3419.0,3419.0,2005-04-04,Ajwan
4,Andhra Pradesh,Kurnool,Adoni,Other,Spices,2.0,3001.0,3001.0,3001.0,2005-04-07,Ajwan
5,Andhra Pradesh,Kurnool,Adoni,Other,Spices,2.0,3306.0,3306.0,3306.0,2005-04-12,Ajwan
6,Andhra Pradesh,Kurnool,Adoni,Other,Spices,3.0,2265.0,3361.0,3298.0,2005-04-13,Ajwan
7,Andhra Pradesh,Kurnool,Adoni,Other,Spices,2.0,2439.0,2439.0,2439.0,2005-04-16,Ajwan
8,Andhra Pradesh,Kurnool,Adoni,Other,Spices,3.0,1401.0,1401.0,1401.0,2005-04-19,Ajwan
9,Andhra Pradesh,Kurnool,Adoni,Other,Spices,1.0,3611.0,3611.0,3611.0,2005-04-20,Ajwan


In [9]:
# Cell 9 — Data quality check on a representative sample

SAMPLE_ROWS = 500_000

quality_sample = pd.read_csv(
    OUTPUT_FILE,
    nrows=SAMPLE_ROWS
)

print("Sample rows:", f"{len(quality_sample):,}")
print("\nMissing values:")
display(quality_sample.isna().sum())

print("\nDuplicate rows in sample:")
print(quality_sample.duplicated().sum())

print("\nData types:")
display(quality_sample.dtypes)

print("\nPrice sanity check:")
print(
    "Negative modal prices:",
    (quality_sample["Modal Price (Rs./Quintal)"] < 0).sum()
)

print(
    "Max < Min:",
    (
        quality_sample["Max Price (Rs./Quintal)"]
        < quality_sample["Min Price (Rs./Quintal)"]
    ).sum()
)

print("\nDate range in sample:")
print("Minimum:", quality_sample["Reported Date"].min())
print("Maximum:", quality_sample["Reported Date"].max())

Sample rows: 500,000

Missing values:


State Name                   0
District Name                0
Market Name                  0
Variety                      0
Group                        0
Arrivals (Tonnes)            0
Min Price (Rs./Quintal)      0
Max Price (Rs./Quintal)      0
Modal Price (Rs./Quintal)    0
Reported Date                0
Commodity                    0
dtype: int64


Duplicate rows in sample:
656

Data types:


State Name                       str
District Name                    str
Market Name                      str
Variety                          str
Group                            str
Arrivals (Tonnes)            float64
Min Price (Rs./Quintal)      float64
Max Price (Rs./Quintal)      float64
Modal Price (Rs./Quintal)    float64
Reported Date                    str
Commodity                        str
dtype: object


Price sanity check:
Negative modal prices: 0
Max < Min: 0

Date range in sample:
Minimum: 2000-10-19
Maximum: 2024-02-02


In [10]:
# Cell 10 — Analyze duplicate market records

DUPLICATE_CHECK_ROWS = 1_000_000

duplicate_sample = pd.read_csv(
    OUTPUT_FILE,
    nrows=DUPLICATE_CHECK_ROWS
)

key_columns = [
    "Commodity",
    "State Name",
    "District Name",
    "Market Name",
    "Variety",
    "Reported Date"
]

duplicate_count = duplicate_sample.duplicated(
    subset=key_columns
).sum()

print(f"Rows checked: {len(duplicate_sample):,}")
print(f"Duplicate market records: {duplicate_count:,}")

if duplicate_count > 0:
    print("\nExample duplicate records:")
    display(
        duplicate_sample[
            duplicate_sample.duplicated(
                subset=key_columns,
                keep=False
            )
        ]
        .sort_values(key_columns)
        .head(20)
    )
else:
    print("No duplicate market records found in sample.")

Rows checked: 1,000,000
Duplicate market records: 37,320

Example duplicate records:


,State Name,District Name,Market Name,Variety,Group,Arrivals (Tonnes),Min Price (Rs./Quintal),Max Price (Rs./Quintal),Modal Price (Rs./Quintal),Reported Date,Commodity
243,Andhra Pradesh,Kurnool,Adoni,Other,Spices,59.0,4609.0,8761.0,7500.0,2010-02-03,Ajwan
244,Andhra Pradesh,Kurnool,Adoni,Other,Spices,59.0,4609.0,8761.0,7500.0,2010-02-03,Ajwan
245,Andhra Pradesh,Kurnool,Adoni,Other,Spices,58.0,5000.0,10051.0,7500.0,2010-02-04,Ajwan
246,Andhra Pradesh,Kurnool,Adoni,Other,Spices,58.0,5000.0,10051.0,7500.0,2010-02-04,Ajwan
247,Andhra Pradesh,Kurnool,Adoni,Other,Spices,111.0,4072.0,9599.0,7000.0,2010-02-05,Ajwan
248,Andhra Pradesh,Kurnool,Adoni,Other,Spices,111.0,4072.0,9599.0,7000.0,2010-02-05,Ajwan
711,Andhra Pradesh,Kurnool,Adoni,Other,Spices,35.0,6786.0,6786.0,6786.0,2015-01-17,Ajwan
712,Andhra Pradesh,Kurnool,Adoni,Other,Spices,35.0,6786.0,6786.0,6786.0,2015-01-17,Ajwan
822,Andhra Pradesh,Kurnool,Kurnool,Other,Spices,0.2,3736.0,3736.0,3736.0,2006-10-10,Ajwan
823,Andhra Pradesh,Kurnool,Kurnool,Other,Spices,0.2,3736.0,3736.0,3736.0,2006-10-10,Ajwan


## Cell 11 — Duplicate analysis (deduplication skipped for performance)

In [ ]:
# Cell 11 — Remove exact duplicate rows

DEDUP_FILE = PROCESSED_DIR / "market_data_deduplicated.csv"

if DEDUP_FILE.exists():
    DEDUP_FILE.unlink()

CHUNK_SIZE = 100_000

total_before = 0
total_after = 0

for chunk in pd.read_csv(
    OUTPUT_FILE,
    chunksize=CHUNK_SIZE
):
    before = len(chunk)

    # Remove only completely identical rows
    chunk = chunk.drop_duplicates()

    after = len(chunk)

    total_before += before
    total_after += after

    chunk.to_csv(
        DEDUP_FILE,
        mode="a",
        header=not DEDUP_FILE.exists(),
        index=False
    )

print("========== DEDUPLICATION COMPLETE ==========")
print(f"Rows before : {total_before:,}")
print(f"Rows after  : {total_after:,}")
print(f"Duplicates removed: {total_before - total_after:,}")
print(f"Output file: {DEDUP_FILE}")

In [12]:
ML_SAMPLE_FILE = PROCESSED_DIR / "ml_modeling_sample.csv"

SAMPLE_SIZE = 1_000_000
RANDOM_STATE = 42

# Read the cleaned dataset in chunks
# and collect a random sample without loading everything into RAM.

samples = []

for chunk in pd.read_csv(
    OUTPUT_FILE,
    chunksize=100_000
):
    sample_fraction = SAMPLE_SIZE / 52_092_498

    samples.append(
        chunk.sample(
            frac=sample_fraction,
            random_state=RANDOM_STATE
        )
    )

    print(f"Processed chunk: {len(chunk):,} rows")

ml_sample = pd.concat(samples, ignore_index=True)

# Make sure we don't exceed the desired size
if len(ml_sample) > SAMPLE_SIZE:
    ml_sample = ml_sample.sample(
        n=SAMPLE_SIZE,
        random_state=RANDOM_STATE
    ).reset_index(drop=True)

ml_sample.to_csv(
    ML_SAMPLE_FILE,
    index=False
)

print("\n========== ML SAMPLE CREATED ==========")
print(f"Rows: {len(ml_sample):,}")
print(f"Columns: {len(ml_sample.columns)}")
print(f"File: {ML_SAMPLE_FILE}")

Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed chunk: 100,000 rows
Processed 

In [13]:
# Cell 13 — Prepare ML modeling dataset

ml_df = pd.read_csv(ML_SAMPLE_FILE)

# Convert date back to datetime
ml_df["Reported Date"] = pd.to_datetime(
    ml_df["Reported Date"],
    errors="coerce"
)

# Sort chronologically within each market/commodity series
ml_df = ml_df.sort_values(
    [
        "Commodity",
        "State Name",
        "District Name",
        "Market Name",
        "Variety",
        "Reported Date"
    ]
).reset_index(drop=True)

print("ML dataset shape:", ml_df.shape)

print("\nDate range:")
print("Minimum:", ml_df["Reported Date"].min())
print("Maximum:", ml_df["Reported Date"].max())

print("\nCommodities:", ml_df["Commodity"].nunique())
print("Markets:", ml_df["Market Name"].nunique())
print("States:", ml_df["State Name"].nunique())

ML dataset shape: (1000000, 11)

Date range:
Minimum: 2001-03-23 00:00:00
Maximum: 2024-02-02 00:00:00

Commodities: 299
Markets: 3198
States: 34


In [ ]:
# Cell 14 — Create time-series price features

# Previous observed model prices
ml_df["Price_Lag_1"] = (
    ml_df.groupby(
        ["Commodity", "Market Name", "Variety"]
    )["Modal Price (Rs./Quintal)"]
    .shift(1)
)

ml_df["Price_Lag_7"] = (
    ml_df.groupby(
        ["Commodity", "Market Name", "Variety"]
    )["Modal Price (Rs./Quintal)"]
    .shift(7)
)

# Rolling historical averages
ml_df["Price_Rolling_7"] = (
    ml_df.groupby(
        ["Commodity", "Market Name", "Variety"]
    )["Modal Price (Rs./Quintal)"]
    .transform(
        lambda x: x.shift(1).rolling(7, min_periods=3).mean()
    )
)

ml_df["Price_Rolling_30"] = (
    ml_df.groupby(
        ["Commodity", "Market Name", "Variety"]
    )["Modal Price (Rs./Quintal)"]
    .transform(
        lambda x: x.shift(1).rolling(30, min_periods=7).mean()
    )
)

# Calendar features
ml_df["Year"] = ml_df["Reported Date"].dt.year
ml_df["Month"] = ml_df["Reported Date"].dt.month
ml_df["Day"] = ml_df["Reported Date"].dt.day
ml_df["DayOfWeek"] = ml_df["Reported Date"].dt.dayofweek

# Price spread
ml_df["Price_Spread"] = (
    ml_df["Max Price (Rs./Quintal)"]
    - ml_df["Min Price (Rs./Quintal)"]
)

# Remove rows where historical features cannot be calculated
ml_df = ml_df.dropna(
    subset=[
        "Price_Lag_1",
        "Price_Lag_7",
        "Price_Rolling_7",
        "Price_Rolling_30"
    ]
).reset_index(drop=True)

print("========== FEATURE ENGINEERING COMPLETE ==========")
print(f"Rows remaining : {len(ml_df):,}")
print(f"Columns         : {len(ml_df.columns)}")

print("\nNew features:")
print([
    "Price_Lag_1",
    "Price_Lag_7",
    "Price_Rolling_7",
    "Price_Rolling_30",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "Price_Spread"
])

print("\nMissing values in new features:")
display(
    ml_df[
        [
            "Price_Lag_1",
            "Price_Lag_7",
            "Price_Rolling_7",
            "Price_Rolling_30"
        ]
    ].isna().sum()
)

========== FEATURE ENGINEERING COMPLETE ==========
Rows remaining : 647,367
Columns         : 20

New features:
['Price_Lag_1', 'Price_Lag_7', 'Price_Rolling_7', 'Price_Rolling_30', 'Year', 'Month', 'Day', 'DayOfWeek', 'Price_Spread']

Missing values in new features:


Price_Lag_1         0
Price_Lag_7         0
Price_Rolling_7     0
Price_Rolling_30    0
dtype: int64

In [15]:
# Cell 15 — Save final ML-ready dataset

FINAL_ML_FILE = PROCESSED_DIR / "ml_ready_data.csv"

ml_df.to_csv(
    FINAL_ML_FILE,
    index=False
)

print("========== FINAL ML DATASET SAVED ==========")
print(f"Rows    : {len(ml_df):,}")
print(f"Columns : {len(ml_df.columns)}")
print(f"File    : {FINAL_ML_FILE}")

========== FINAL ML DATASET SAVED ==========
Rows    : 647,367
Columns : 20
File    : ..\data\processed\ml_ready_data.csv
